# Chapter 5 — Severity and robustness

Companion notebook for *Evaluating Large Language Models in High-Stakes Domains* (Costa, 2026). All data in this notebook are simulated; the generating process is in the code and described in the text.

In [ ]:
"""Figures and worked-example numbers for 'Evaluating Large Language Models in High-Stakes Domains'.
All data are analytic or simulated; generating processes are stated in the text.
Run:  python figures.py   (writes PNGs and results.json into ./figures)
"""
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, minimize_scalar

OUT = os.path.join(os.getcwd(), "figures_out")
os.makedirs(OUT, exist_ok=True)

# palette (categorical, fixed order) and ink
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
    "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.dpi": 200, "savefig.dpi": 200,
    "savefig.facecolor": "white",
})
results = {}
se = lambda x: x.std(ddof=1)/np.sqrt(len(x))
def wilson(x, n, z=1.96):
    p = x/n; c = (p + z*z/(2*n))/(1 + z*z/n)
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/(1 + z*z/n)
    return c-h, c+h
def clopper_upper(x, n, conf=0.95):
    return 1.0 if x == n else stats.beta.ppf(conf, x+1, n-x)


## Figure 6 + Table 5.1: severity and perturbation (Ch. 5)

In [ ]:
rng = np.random.default_rng(55)
n5 = 200
sev_models = ["Model X", "Model Y", "Model Z"]
err_rate = {"Model X": 0.10, "Model Y": 0.15, "Model Z": 0.25}
# severity distribution among errors (0 none, 1 minor, 2 moderate, 3 severe)
sev_dist = {"Model X": [0.30, 0.30, 0.20, 0.20], "Model Y": [0.50, 0.30, 0.15, 0.05], "Model Z": [0.60, 0.30, 0.08, 0.02]}
cost_w = np.array([0, 1, 5, 25])     # illustrative harm weights per severity level
rows5 = []; sev_counts = {}
for m in sev_models:
    e = rng.binomial(1, err_rate[m], n5)
    v = np.where(e == 1, rng.choice(4, n5, p=sev_dist[m]), 0)
    counts = np.array([((e == 1) & (v == k)).sum() for k in range(4)]); sev_counts[m] = counts
    n_err = int(e.sum()); n_sev = int(counts[3])
    exp_sev = v.mean(); exp_harm = cost_w[v].mean()
    lo_e, hi_e = wilson(n_err, n5)
    rows5.append(f"| {m} | {n_err/n5:.3f} ({lo_e:.3f}–{hi_e:.3f}) | {counts[0]}/{counts[1]}/{counts[2]}/{counts[3]} | {exp_sev:.3f} | {n_sev} ({100*clopper_upper(n_sev, n5):.1f}%) | {exp_harm:.2f} |")
results["ch5"] = {"SEV_TABLE": "\n".join(rows5)}
# perturbation: paraphrase each item; two models with the same mean change but different flip rates
pert = {}
for m, p_corr, flip in [("Model X", 0.85, 0.06), ("Model Z", 0.75, 0.22)]:
    base_c = rng.binomial(1, p_corr, n5)
    do_flip = rng.binomial(1, flip, n5)
    pert_c = np.where(do_flip == 1, 1 - base_c, base_c)
    d = pert_c - base_c
    lo_f, hi_f = wilson(int(do_flip.sum()), n5)
    pert[m] = dict(flip=do_flip.mean(), flip_ci=(lo_f, hi_f), change=d.mean(), change_se=se(d.astype(float)), base=base_c.mean(), after=pert_c.mean())
results["ch5"].update({
    "FLIP_X": f"{100*pert['Model X']['flip']:.1f}% ({100*pert['Model X']['flip_ci'][0]:.1f}–{100*pert['Model X']['flip_ci'][1]:.1f}%)",
    "FLIP_Z": f"{100*pert['Model Z']['flip']:.1f}% ({100*pert['Model Z']['flip_ci'][0]:.1f}–{100*pert['Model Z']['flip_ci'][1]:.1f}%)",
    "CHG_X": f"{pert['Model X']['change']:+.3f} ({pert['Model X']['change']-1.96*pert['Model X']['change_se']:+.3f}, {pert['Model X']['change']+1.96*pert['Model X']['change_se']:+.3f})",
    "CHG_Z": f"{pert['Model Z']['change']:+.3f} ({pert['Model Z']['change']-1.96*pert['Model Z']['change_se']:+.3f}, {pert['Model Z']['change']+1.96*pert['Model Z']['change_se']:+.3f})",
    "ACC_X": f"{pert['Model X']['base']:.3f}", "ACC_Z": f"{pert['Model Z']['base']:.3f}",
    "ERR_X": f"{err_rate['Model X']:.2f}", "ERR_Z": f"{err_rate['Model Z']:.2f}",
})
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9), gridspec_kw={"width_ratios": [1.2, 1]})
ramp = ["#c9ddf5", "#7fb0e8", "#2a78d6", "#123f78"]       # one hue, light -> dark (ordinal severity)
left = np.zeros(len(sev_models))
for k in range(4):
    vals = np.array([sev_counts[m][k] for m in sev_models])/n5*100
    ax[0].barh(sev_models, vals, left=left, color=ramp[k], edgecolor="white", linewidth=1.5, label=["no harm", "minor", "moderate", "severe"][k])
    left += vals
ax[0].invert_yaxis(); ax[0].set_xlabel("Errors as % of items, by severity"); ax[0].legend(fontsize=7.5, loc="upper right", ncol=2)
ax[0].set_title("(a) Error rate and severity (200 items)", loc="left", fontsize=9, color=INK)
labels = ["Model X", "Model Z"]
ax[1].errorbar([100*pert[m]["flip"] for m in labels], [0, 1], xerr=[[100*(pert[m]["flip"]-pert[m]["flip_ci"][0]) for m in labels], [100*(pert[m]["flip_ci"][1]-pert[m]["flip"]) for m in labels]],
               fmt="o", color=C2, ecolor=C2, capsize=3, ms=6, lw=1.5, label="flip rate (% of items)")
ax[1].errorbar([100*pert[m]["change"] for m in labels], [0.25, 1.25], xerr=[100*1.96*pert[m]["change_se"] for m in labels],
               fmt="s", color=C1, ecolor=C1, capsize=3, ms=6, lw=1.5, label="mean change in accuracy (points)")
ax[1].axvline(0, color=INK2, ls="--", lw=1); ax[1].set_yticks([0.1, 1.1]); ax[1].set_yticklabels(labels); ax[1].invert_yaxis()
ax[1].set_xlabel("Under paraphrase perturbation"); ax[1].legend(fontsize=7, loc="upper right"); ax[1].set_xlim(-12, 36)
ax[1].set_title("(b) Robustness: flips vs. mean change", loc="left", fontsize=9, color=INK)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig6_severity_robustness.png")); plt.close(fig)

with open(os.path.join(OUT, "results.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(json.dumps(results, indent=2, ensure_ascii=True))

In [ ]:
import json
print(json.dumps(results, indent=2, ensure_ascii=False))